In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
BASE_URL = "https://gpustack.cds.utsa.edu/v1" # the class model server address 
API_KEY = "REPLACE-WITH-YOUR-KEY" # your personal API key 
client = OpenAI(base_url=BASE_URL,
api_key=API_KEY) # Ask the server which model it is serving, and remember its name: MODEL =
client.models.list().data[1].id 
print("Connected. Model:", MODEL)

NameError: name 'OpenAI' is not defined

In [5]:
%pip install openai


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
from openai import OpenAI
BASE_URL = "https://gpustack.cds.utsa.edu/v1" # the class model server address 
API_KEY = "REPLACE-WITH-YOUR-KEY" # your personal API key 
client = OpenAI(base_url=BASE_URL,
api_key=API_KEY) # Ask the server which model it is serving, and remember its name: 
MODEL = client.models.list().data[1].id 
print("Connected. Model:", MODEL)

Connected. Model: qwen3.5-122b-a10b


In [19]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""

    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )

            return resp.choices[0].message.content

        except Exception:
            if attempt == retries:
                raise

            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))



Hello!


In [20]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM "
    "to classify a security log line and returns a dataclass with label "
    "and reason. Suggest a robust prompt format and a defensive way to "
    "parse the reply. Keep it concise."
)

print(ask_model(question))



```python
from dataclasses import dataclass
import json
import re

@dataclass
class LogTriage:
    label: str  # e.g., "benign", "suspicious", "malicious"
    reason: str

PROMPT = """
Analyze this security log line. Classify it into one of: ['benign', 'suspicious', 'malicious', 'unknown'].
Respond ONLY with a valid JSON object matching this schema:
{{"label": "string", "reason": "string"}}

Log Line: "{log_line}"
"""

def triage_log(line: str, client) -> LogTriage:
    try:
        # 1. Call LLM (assuming generic completions API)
        response = client.chat.completions.create(
            model="security-model",
            messages=[{"role": "user", "content": PROMPT.format(log_line=line)}],
            temperature=0
        )
        raw_content = response.choices[0].message.content

        # 2. Defensive JSON Extraction (handle markdown/code blocks)
        json_match = re.search(r'\{.*\}', raw_content, re.DOTALL)
        if not json_match:
            raise ValueError("No JS

In [22]:
from dataclasses import dataclass
from enum import Enum
import re


class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"


@dataclass
class Triage:
    line: str
    label: Label
    reason: str


LABEL_RE = re.compile(
    r"^LABEL:\s*(\w+)",
    re.IGNORECASE | re.MULTILINE
)

REASON_RE = re.compile(
    r"^REASON:\s*(.+)",
    re.IGNORECASE | re.MULTILINE
)

In [23]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label[raw] if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(
        line=line,
        label=label,
        reason=reason
    )

In [24]:
TOOL_NAME = "LogHero" # change this to your own creative name

In [25]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='A failed SSH authentication attempt for the root account from an external IP address suggests a potential brute-force attack or unauthorized access attempt.')


In [26]:
LOG_FILE = "security_log.txt"  # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20  # keep this first version small and quick


def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    return lines[:limit]


entries = read_log_lines(LOG_FILE)

results = [triage_log(line) for line in entries]

print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.txt


from datetime import datetime


def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")

        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")

    return path


report_path = write_report(results, TOOL_NAME)

print("Wrote", report_path, "\n")

with open(report_path, encoding="utf-8") as f:
    print(f.read())